# ThriftAI: cut multi-agent LLM costs by 56% during prompt iteration

Multi-agent LLM pipelines are expensive to develop because every prompt tweak re-runs the *entire* pipeline. ThriftAI sits between your code and the LLM, caches per-agent responses, and lets you replay N&minus;1 agents from a previous trace while iterating on just one.

Below: a research pipeline (`researcher` &rarr; `analyzer` &rarr; `writer`) run three times — cold, warm, and during a writer-prompt tweak. Watch the cost.

In [ ]:
import getpass
import os
import pathlib
import shutil

import matplotlib.pyplot as plt

import thriftai as ta

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

In [ ]:
MODEL = "anthropic/claude-haiku-4-5"  # cheap; full demo costs <$0.05

# Start from a clean cache so the demo is deterministic on re-runs.
demo_dir = pathlib.Path("./.thriftai-demo")
if demo_dir.exists():
    shutil.rmtree(demo_dir)


@ta.agent(name="researcher")
def research(s, topic):
    return s.completion(
        messages=[
            {"role": "system", "content":
                "You are a research assistant. Be concise. "
                "Return a short paragraph of factual context."},
            {"role": "user", "content": f"Research this topic: {topic}"},
        ],
        model=MODEL,
    )


@ta.agent(name="analyzer", depends_on=["researcher"])
def analyze(s, raw):
    return s.completion(
        messages=[
            {"role": "system", "content":
                "You are an analyst. List the 3 most important "
                "insights from the research, one per line."},
            {"role": "user", "content": f"Research:\n\n{raw}"},
        ],
        model=MODEL,
    )


@ta.agent(name="writer", depends_on=["analyzer"])
def write(s, analysis, *, style="executive"):
    return s.completion(
        messages=[
            {"role": "system", "content":
                f"You are a technical writer. Produce a {style} summary "
                "in 3 short paragraphs."},
            {"role": "user", "content": f"Analysis:\n\n{analysis}"},
        ],
        model=MODEL,
    )


session = ta.Session(cache_dir=str(demo_dir))
TOPIC = "The impact of AI on cybersecurity"
runs = []  # (label, total_cost) pairs we'll chart at the end

## Run 1 — Cold

All three agents go live. This is the baseline cost of running the pipeline once.

In [ ]:
with session.run() as run:
    raw = research(run, TOPIC)
    analysis = analyze(run, raw)
    summary = write(run, analysis, style="executive")
    trace_id_1 = run.trace_id

print(run.cost_report.summary())
runs.append(("1: cold", run.cost_report.total_cost))

## Run 2 — Re-run with no changes

Same inputs. The exact-match cache catches all three agents. Cost should be ≈ $0.

In [ ]:
with session.run() as run:
    raw = research(run, TOPIC)
    analysis = analyze(run, raw)
    summary = write(run, analysis, style="executive")

print(run.cost_report.summary())
runs.append(("2: re-run", run.cost_report.total_cost))

## Run 3 — Tweak the writer's style

We change just the writer's `style` from `"executive"` to `"conversational"`. With **selective replay**, the researcher and analyzer return their cached outputs; only the writer goes live.

This is the core ThriftAI loop: iterate on one agent without re-paying for the rest.

In [ ]:
with session.replay(trace_id=trace_id_1, live=["writer"]) as run:
    raw = research(run, TOPIC)
    analysis = analyze(run, raw)
    summary = write(run, analysis, style="conversational")

print(run.cost_report.summary())
runs.append(("3: writer fix", run.cost_report.total_cost))
print("\n--- Final summary ---\n")
print(summary)

In [ ]:
labels = [r[0] for r in runs]
values = [r[1] for r in runs]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=["#cf4647", "#5fb978", "#3585b8"])
ax.set_ylabel("Run cost (USD)")
ax.set_title("Cost per pipeline run with ThriftAI")
for bar, v in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, v,
        f"${v:.4f}", ha="center", va="bottom", fontsize=10,
    )
ax.set_ylim(0, max(max(values) * 1.2, 0.001))
plt.tight_layout()
plt.show()

without_thriftai = values[0] * 3  # if all three runs went live
total_with = sum(values)
saved = without_thriftai - total_with
pct = (saved / without_thriftai * 100) if without_thriftai > 0 else 0
print(f"Total spent across 3 runs: ${total_with:.4f}")
print(f"Cost without ThriftAI:     ${without_thriftai:.4f}")
print(f"Saved:                     ${saved:.4f}  ({pct:.0f}%)")

## Take-away

In a typical iteration loop, only one or two agents change between runs. ThriftAI catches everything else, so each iteration costs a fraction of the full-pipeline price.

**Try it on your own pipeline:**

```bash
pip install thriftai
```

```python
import thriftai as ta

@ta.agent(name="my_agent", depends_on=["upstream"])
def my_agent(session, x):
    return session.completion(messages=[...], model="...")

session = ta.Session()
with session.run() as run:
    ...

# Later, after tweaking just `my_agent`'s prompt:
with session.replay(trace_id=run.trace_id, live=["my_agent"]) as run:
    ...
    print(run.cost_report.summary())
```

ThriftAI is open source under MIT. See the [README](../README.md) for the full feature set, including semantic caching and the production-mode kill switch.